# Inventario de Drive y plan de carga a GCS

Este notebook inventaria datasets almacenados en Google Drive y genera un plan exacto de carga hacia Google Cloud Storage. Es solo una herramienta de inventario y planificacion: no realiza cargas, no autentica Google Cloud, no modifica datos originales y no ejecuta entrenamiento.

Alcance:

- SPIDER se toma desde las carpetas extraidas `images/images` y `masks/masks`.
- AXIAL se limita a los archivos referenciados por el CSV E9.
- ZIP, XCF, MAT, XLSX, documentacion y archivos auxiliares no forman parte del plan.
- El modelo sagital de bootstrap se inspecciona, pero su inclusion es opcional.
- El JSON E5 es la fuente principal del label mapping.
- Los resultados opcionales se guardan en una carpeta nueva de planificacion bajo Drive.
- No se modifican los datos originales.


## Modo de uso seguro

Ejecutar primero con `WRITE_PLAN_FILES = False`. En ese modo no se crea `PLAN_ROOT` ni se escribe ningun archivo; solo se muestran tablas y validaciones.

Cuando el resumen sea correcto, cambiar `WRITE_PLAN_FILES = True` para guardar manifests y resumenes dentro de `results/GCS_dataset_migration_plan`. Este notebook sigue sin cargar archivos a GCS.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import pandas as pd


## Montaje seguro de Drive

La celda reutiliza `/content/drive/MyDrive` si ya esta disponible. Si no existe, intenta montar Drive sin forzar remount. Si el mountpoint queda ocupado de forma incompatible, falla con un mensaje claro.


In [2]:
DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"

try:
    from google.colab import drive  # type: ignore
except ModuleNotFoundError:
    drive = None

if MY_DRIVE.exists():
    print(f"Drive ya disponible: {MY_DRIVE}")
elif drive is None:
    raise RuntimeError("Este notebook esta pensado para Colab. No se encontro google.colab y Drive no esta montado.")
elif DRIVE_ROOT.exists() and any(DRIVE_ROOT.iterdir()):
    raise RuntimeError("/content/drive existe pero MyDrive no esta disponible. Revisar el mountpoint manualmente; no se limpia automaticamente.")
else:
    drive.mount(str(DRIVE_ROOT))

PFI_ROOT_CHECK = MY_DRIVE / "PFI_MVP"
if not PFI_ROOT_CHECK.exists():
    raise FileNotFoundError(f"No existe {PFI_ROOT_CHECK}. Verificar ubicacion del proyecto en Drive.")
print(f"PFI_MVP encontrado: {PFI_ROOT_CHECK}")


Mounted at /content/drive
PFI_MVP encontrado: /content/drive/MyDrive/PFI_MVP


## Configuracion de rutas


In [3]:
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")

SPIDER_IMAGES_SOURCE = PFI_ROOT / "data/SPIDER/images/images"
SPIDER_MASKS_SOURCE = PFI_ROOT / "data/SPIDER/masks/masks"
AXIAL_ROOT = PFI_ROOT / "data/AXIAL_ALKAFRI"
E5_MAPPING_JSON = PFI_ROOT / "results/E5_multiclase_agrupado/E5_multiclass_label_mapping.json"
E9_SPLIT_CSV = PFI_ROOT / "results/E9_alkafri_axial_t2_final_labels_baseline/E9_t2_final_labels_curated_split.csv"
SAGITTAL_BOOTSTRAP_MODEL = PFI_ROOT / "models/final/sagittal_spider_multiclass_final_best.pt"
PLAN_ROOT = PFI_ROOT / "results/GCS_dataset_migration_plan"

GCS_BUCKET_URI = "gs://pfi-rm-lumbar-artifacts-2026-ef"
SPIDER_IMAGES_DEST = "gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/SPIDER/images"
SPIDER_MASKS_DEST = "gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/SPIDER/masks"
AXIAL_DEST = "gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/AXIAL_ALKAFRI"
METADATA_DEST = "gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/metadata"
BOOTSTRAP_DEST = "gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/bootstrap_models"

WRITE_PLAN_FILES = False
COMPUTE_FULL_SHA256 = False
INCLUDE_BOOTSTRAP_MODEL = False

ALLOWED_DATA_SUFFIXES = {".mha", ".mhd", ".npy", ".png", ".tif", ".tiff", ".dcm", ".ima"}
EXCLUDED_SUFFIXES = {".zip", ".xcf", ".mat", ".xlsx", ".xls", ".docx", ".pdf", ".txt", ".csv", ".json", ".py", ".ipynb"}
AXIAL_COLUMNS = ["image_file_path", "final_label_file_path", "case_id_norm", "split"]


## Helpers de inventario


In [4]:
def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def file_modified_at(path: Path) -> str | None:
    if not path.exists():
        return None
    return datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def sha256_stream(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def gcs_join(base: str, relative_path: str | Path) -> str:
    rel = str(relative_path).replace("\\", "/").lstrip("/")
    return f"{base.rstrip('/')}/{rel}"


def path_type(path: Path) -> str:
    if path.is_dir():
        return "dir"
    if path.is_file():
        return "file"
    if path.exists():
        return "other"
    return "missing"


def is_readable(path: Path) -> bool:
    try:
        if path.is_dir():
            next(path.iterdir(), None)
        elif path.is_file():
            with open(path, "rb") as fh:
                fh.read(1)
        else:
            return False
        return True
    except Exception:
        return False


def validation_table(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for key, path in paths.items():
        rows.append({
            "key": key,
            "path": str(path),
            "exists": path.exists(),
            "type": path_type(path),
            "readable": is_readable(path),
            "size_bytes": path.stat().st_size if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def safe_relative_path(path: Path, root: Path) -> Path:
    resolved = path.resolve()
    base = root.resolve()
    rel = resolved.relative_to(base)
    if any(part in {"", ".", ".."} for part in rel.parts):
        raise ValueError(f"relative_path inseguro: {rel}")
    return rel


def plan_row(component: str, source_path: Path, source_root: Path, destination_base: str, *, referenced_rows: int | None = None, sha256: str | None = None) -> dict[str, Any]:
    exists = source_path.exists()
    suffix = source_path.suffix.lower()
    is_symlink = source_path.is_symlink()
    size = source_path.stat().st_size if source_path.is_file() else None
    status = "OK"
    rel_text = ""
    destination_uri = ""
    try:
        rel = safe_relative_path(source_path, source_root)
        rel_text = rel.as_posix()
        destination_uri = gcs_join(destination_base, rel)
    except Exception as exc:
        status = f"INVALID_RELATIVE_PATH: {exc}"
    if not exists:
        status = "MISSING"
    elif is_symlink:
        status = "SYMLINK"
    elif size == 0:
        status = "ZERO_SIZE"
    elif suffix in EXCLUDED_SUFFIXES:
        status = "EXCLUDED_SUFFIX"
    elif suffix not in ALLOWED_DATA_SUFFIXES and component not in {"metadata_e5", "metadata_e9", "bootstrap_model"}:
        status = "UNSUPPORTED_SUFFIX"
    return {
        "component": component,
        "source_path": str(source_path),
        "source_root": str(source_root),
        "relative_path": rel_text,
        "destination_uri": destination_uri,
        "suffix": suffix,
        "size_bytes": size,
        "modified_at": file_modified_at(source_path),
        "sha256": sha256,
        "referenced_rows": referenced_rows,
        "exists": exists,
        "is_symlink": is_symlink,
        "status": status,
    }


## Validacion inicial de rutas


In [5]:
required_paths = {
    "SPIDER_IMAGES_SOURCE": SPIDER_IMAGES_SOURCE,
    "SPIDER_MASKS_SOURCE": SPIDER_MASKS_SOURCE,
    "AXIAL_ROOT": AXIAL_ROOT,
    "E5_MAPPING_JSON": E5_MAPPING_JSON,
    "E9_SPLIT_CSV": E9_SPLIT_CSV,
    "SAGITTAL_BOOTSTRAP_MODEL": SAGITTAL_BOOTSTRAP_MODEL,
}
path_df = validation_table(required_paths)
display(path_df)

required_failures = []
for key in ["SPIDER_IMAGES_SOURCE", "SPIDER_MASKS_SOURCE", "AXIAL_ROOT"]:
    if not required_paths[key].is_dir():
        required_failures.append(f"{key} debe existir y ser directorio")
for key in ["E5_MAPPING_JSON", "E9_SPLIT_CSV"]:
    if not required_paths[key].is_file():
        required_failures.append(f"{key} debe existir y ser archivo")

print(f"SPIDER images/images encontrado: {SPIDER_IMAGES_SOURCE.exists()}")
print(f"SPIDER masks/masks encontrado: {SPIDER_MASKS_SOURCE.exists()}")
print("No se utilizaran images.zip ni masks.zip.")
print(f"Bootstrap sagital presente: {SAGITTAL_BOOTSTRAP_MODEL.is_file()}")

if required_failures:
    raise FileNotFoundError("; ".join(required_failures))


,key,path,exists,type,readable,size_bytes
0,SPIDER_IMAGES_SOURCE,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,True,dir,True,NaN
1,SPIDER_MASKS_SOURCE,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,True,dir,True,NaN
2,AXIAL_ROOT,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,True,dir,True,NaN
3,E5_MAPPING_JSON,/content/drive/MyDrive/PFI_MVP/results/E5_mult...,True,file,True,222.0
4,E9_SPLIT_CSV,/content/drive/MyDrive/PFI_MVP/results/E9_alka...,True,file,True,422594.0
5,SAGITTAL_BOOTSTRAP_MODEL,/content/drive/MyDrive/PFI_MVP/models/final/sa...,True,file,True,1975947.0


SPIDER images/images encontrado: True
SPIDER masks/masks encontrado: True
No se utilizaran images.zip ni masks.zip.
Bootstrap sagital presente: True


## Inventario SPIDER


In [6]:
def inventory_tree(component: str, source_root: Path, destination_base: str) -> pd.DataFrame:
    rows = []
    for path in sorted(source_root.rglob("*")):
        if not path.is_file():
            continue
        suffix = path.suffix.lower()
        if suffix in EXCLUDED_SUFFIXES:
            continue
        if suffix not in ALLOWED_DATA_SUFFIXES:
            continue
        rows.append(plan_row(component, path, source_root, destination_base))
    return pd.DataFrame(rows)


def normalized_pair_stem(path_like: str | Path) -> str:
    stem = Path(path_like).stem.lower()
    for suffix in ["_image", "_mask", "_label"]:
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
    return stem


def spider_pairing_summary(images_df: pd.DataFrame, masks_df: pd.DataFrame) -> dict[str, Any]:
    image_stems = {normalized_pair_stem(p) for p in images_df.get("relative_path", [])}
    mask_stems = {normalized_pair_stem(p) for p in masks_df.get("relative_path", [])}
    shared = image_stems & mask_stems
    return {
        "images_total": len(image_stems),
        "masks_total": len(mask_stems),
        "shared_stems": len(shared),
        "images_without_mask": sorted(image_stems - mask_stems)[:20],
        "masks_without_image": sorted(mask_stems - image_stems)[:20],
        "ready": len(shared) > 0,
    }

spider_images_df = inventory_tree("spider_image", SPIDER_IMAGES_SOURCE, SPIDER_IMAGES_DEST)
spider_masks_df = inventory_tree("spider_mask", SPIDER_MASKS_SOURCE, SPIDER_MASKS_DEST)
spider_pairing = spider_pairing_summary(spider_images_df, spider_masks_df)

print("SPIDER images por extension:", dict(Counter(spider_images_df.get("suffix", []))))
print("SPIDER masks por extension:", dict(Counter(spider_masks_df.get("suffix", []))))
print("SPIDER pairing:", spider_pairing)
display(spider_images_df.head(10))
display(spider_masks_df.head(10))


SPIDER images por extension: {'.mha': 447}
SPIDER masks por extension: {'.mha': 447}
SPIDER pairing: {'images_total': 447, 'masks_total': 447, 'shared_stems': 447, 'images_without_mask': [], 'masks_without_image': [], 'ready': True}


,component,source_path,source_root,relative_path,destination_uri,suffix,size_bytes,modified_at,sha256,referenced_rows,exists,is_symlink,status
0,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,100_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,7105555,2022-11-01T15:19:38Z,None,None,True,False,OK
1,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,100_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,7118571,2022-11-01T15:19:36Z,None,None,True,False,OK
2,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,101_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,1839721,2022-11-01T15:12:26Z,None,None,True,False,OK
3,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,101_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2361432,2022-11-01T15:12:24Z,None,None,True,False,OK
4,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,104_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,1822712,2022-11-01T15:08:20Z,None,None,True,False,OK
5,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,104_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2706068,2022-11-01T15:08:18Z,None,None,True,False,OK
6,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,105_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,4534255,2022-11-01T15:03:52Z,None,None,True,False,OK
7,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,105_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,4604875,2022-11-01T15:03:52Z,None,None,True,False,OK
8,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,106_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2971403,2022-11-01T15:13:34Z,None,None,True,False,OK
9,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,106_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2912377,2022-11-01T15:13:34Z,None,None,True,False,OK


,component,source_path,source_root,relative_path,destination_uri,suffix,size_bytes,modified_at,sha256,referenced_rows,exists,is_symlink,status
0,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,100_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,187098,2023-01-23T11:53:58Z,None,None,True,False,OK
1,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,100_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,187098,2023-01-23T11:52:22Z,None,None,True,False,OK
2,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,101_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,59612,2023-01-23T11:49:20Z,None,None,True,False,OK
3,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,101_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,76227,2023-01-23T11:55:24Z,None,None,True,False,OK
4,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,104_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,53075,2023-01-23T11:58:44Z,None,None,True,False,OK
5,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,104_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,72849,2023-01-23T11:48:16Z,None,None,True,False,OK
6,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,105_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,105689,2023-01-23T11:45:02Z,None,None,True,False,OK
7,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,105_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,105689,2023-01-23T11:58:58Z,None,None,True,False,OK
8,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,106_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,73639,2023-01-23T12:00:30Z,None,None,True,False,OK
9,spider_mask,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,106_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,73832,2023-01-23T11:46:34Z,None,None,True,False,OK


## Validacion del mapping E5


In [7]:
def normalize_label_mapping(raw: dict[str, Any], *, num_classes: int = 4) -> dict[int, list[int]]:
    if not isinstance(raw, dict) or not raw:
        raise ValueError("mapping E5 debe ser un dict no vacio")
    normalized: dict[int, set[int]] = defaultdict(set)
    for key, value in raw.items():
        try:
            key_int = int(key)
        except Exception as exc:
            raise ValueError(f"clave no numerica: {key!r}") from exc
        if isinstance(value, list):
            class_id = key_int
            raw_labels = [int(v) for v in value]
        else:
            raw_labels = [key_int]
            class_id = int(value)
        if class_id < 0 or class_id >= num_classes:
            raise ValueError(f"class_id fuera de rango 0..{num_classes-1}: {class_id}")
        for raw_label in raw_labels:
            normalized[class_id].add(int(raw_label))
    if 0 not in normalized:
        normalized[0].add(0)
    return {class_id: sorted(labels) for class_id, labels in sorted(normalized.items())}

with open(E5_MAPPING_JSON, "r", encoding="utf-8") as fh:
    e5_raw_mapping = json.load(fh)
e5_mapping_normalized = normalize_label_mapping(e5_raw_mapping, num_classes=4)
e5_sha256 = sha256_stream(E5_MAPPING_JSON)
metadata_e5_row = plan_row("metadata_e5", E5_MAPPING_JSON, E5_MAPPING_JSON.parent, METADATA_DEST, sha256=e5_sha256)
metadata_e5_row["relative_path"] = "E5_multiclass_label_mapping.json"
metadata_e5_row["destination_uri"] = gcs_join(METADATA_DEST, "E5_multiclass_label_mapping.json")

print("E5 size_bytes:", E5_MAPPING_JSON.stat().st_size)
print("E5 sha256:", e5_sha256)
print("E5 entradas:", len(e5_raw_mapping))
print("E5 mapping normalizado:", e5_mapping_normalized)
print("E5 raw labels:", sorted({label for labels in e5_mapping_normalized.values() for label in labels}))
print("E5 clases finales:", sorted(e5_mapping_normalized))


E5 size_bytes: 222
E5 sha256: c02d07b6c4c4499af9bc42e581bda00021972466bee646c5d7174a64fafafa12
E5 entradas: 20
E5 mapping normalizado: {0: [0], 1: [1, 2, 3, 4, 5, 6, 7, 8, 9], 2: [100], 3: [201, 202, 203, 204, 205, 206, 207, 208, 209]}
E5 raw labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 100, 201, 202, 203, 204, 205, 206, 207, 208, 209]
E5 clases finales: [0, 1, 2, 3]


## Inventario axial dirigido por E9

Se intenta importar `resolve_portable_axial_path` desde el repo. Si no esta disponible, se usa una implementacion equivalente en esta celda, que debe mantenerse en paridad con `cloud_runtime.py`.


In [8]:
def _fallback_resolve_portable_axial_path(raw_value: object, *, primary_root: Path, additional_roots: list[Path], anchor: str = "AXIAL_ALKAFRI") -> tuple[Path, bool]:
    if raw_value is None or (isinstance(raw_value, float) and math.isnan(raw_value)):
        raise ValueError("path axial nulo")
    text = str(raw_value).strip()
    if not text:
        raise ValueError("path axial vacio")
    normalized = text.replace("\\", "/")
    direct = Path(text)
    if direct.is_absolute() and direct.exists():
        return direct, False
    roots = [primary_root, *additional_roots]
    if not direct.is_absolute():
        for root in roots:
            candidate = root / direct
            if candidate.exists():
                return candidate, False
    parts = normalized.split("/")
    anchor_index = next((i for i, part in enumerate(parts) if part.lower() == anchor.lower()), None)
    if anchor_index is not None:
        suffix = "/".join(parts[anchor_index + 1:]).lstrip("/")
        if not suffix or any(part in {"", ".", ".."} for part in suffix.split("/")):
            raise ValueError(f"suffix axial inseguro: {suffix!r}")
        return primary_root / Path(*suffix.split("/")), True
    if not direct.is_absolute():
        return primary_root / direct, False
    raise ValueError(f"no se pudo interpretar path axial: {raw_value!r}")

try:
    import sys
    candidate_repo = Path("/content/PFI_MVPTest_Enzo_AImodule")
    if candidate_repo.exists():
        sys.path.insert(0, str(candidate_repo / "ai_service"))
    from pfi_ai_service.training.cloud_runtime import resolve_portable_axial_path as _repo_resolver
    def resolve_portable_axial_path(raw_value: object, *, primary_root: Path, additional_roots: list[Path], anchor: str = "AXIAL_ALKAFRI") -> tuple[Path, bool]:
        return _repo_resolver(raw_value, primary_root=primary_root, additional_roots=additional_roots, anchor=anchor)
    RESOLVER_SOURCE = "repo cloud_runtime.py"
except Exception:
    resolve_portable_axial_path = _fallback_resolve_portable_axial_path
    RESOLVER_SOURCE = "fallback notebook"

print("Resolver axial:", RESOLVER_SOURCE)


def validate_e9_split(df: pd.DataFrame) -> dict[str, Any]:
    missing_cols = [col for col in AXIAL_COLUMNS if col not in df.columns]
    if missing_cols:
        raise KeyError(f"faltan columnas E9: {missing_cols}")
    if len(df) == 0:
        raise ValueError("CSV E9 sin filas")
    if df[["image_file_path", "final_label_file_path", "case_id_norm", "split"]].isna().any().any():
        raise ValueError("CSV E9 contiene nulos en columnas requeridas")
    split_values = set(df["split"].astype(str).str.lower())
    if not split_values.issubset({"train", "val", "test"}):
        raise ValueError(f"splits inesperados: {sorted(split_values)}")
    if split_values != {"train", "val", "test"}:
        raise ValueError(f"faltan splits: {sorted({'train','val','test'} - split_values)}")
    patient_splits = df.groupby("case_id_norm")["split"].nunique()
    mixed_patients = patient_splits[patient_splits > 1]
    if len(mixed_patients):
        raise ValueError(f"pacientes mezclados entre splits: {mixed_patients.index.tolist()[:20]}")
    duplicates = int(df.duplicated(["image_file_path", "final_label_file_path"]).sum())
    by_split = df.groupby("split").agg(rows=("split", "size"), patients=("case_id_norm", "nunique")).reset_index()
    return {"duplicates": duplicates, "by_split": by_split}


def axial_relative_from_root(path: Path) -> Path:
    return safe_relative_path(path, AXIAL_ROOT)


def build_axial_plan(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    records = []
    preview_rows = []
    problems: dict[str, list[str]] = defaultdict(list)
    references: dict[tuple[str, str], int] = Counter()
    rebased = 0
    not_rebased = 0
    for idx, row in df.iterrows():
        for component, column in [("axial_image", "image_file_path"), ("axial_mask", "final_label_file_path")]:
            try:
                resolved, was_rebased = resolve_portable_axial_path(row[column], primary_root=AXIAL_ROOT, additional_roots=[E9_SPLIT_CSV.parent, AXIAL_ROOT])
            except Exception as exc:
                problems[f"{component}_resolve"].append(f"row={idx}: {exc}")
                continue
            if was_rebased:
                rebased += 1
            else:
                not_rebased += 1
            references[(component, str(resolved))] += 1
            preview_rows.append({"row": idx, "component": component, "raw_path": row[column], "resolved_path": str(resolved), "was_rebased": was_rebased})
    for (component, resolved_text), count in sorted(references.items()):
        path = Path(resolved_text)
        row = plan_row(component, path, AXIAL_ROOT, AXIAL_DEST, referenced_rows=count)
        try:
            rel = axial_relative_from_root(path)
            row["relative_path"] = rel.as_posix()
            row["destination_uri"] = gcs_join(AXIAL_DEST, rel)
        except Exception as exc:
            row["status"] = f"INVALID_RELATIVE_PATH: {exc}"
        records.append(row)
    plan = pd.DataFrame(records)
    preview = pd.DataFrame(preview_rows)
    stats = {"paths_total": len(preview_rows), "paths_rebased": rebased, "paths_not_rebased": not_rebased, "problems": dict(problems)}
    return plan, preview, stats


e9_df = pd.read_csv(E9_SPLIT_CSV)
e9_validation = validate_e9_split(e9_df)
print("E9 filas:", len(e9_df))
print("E9 duplicados exactos de pares:", e9_validation["duplicates"])
display(e9_validation["by_split"])

axial_plan_df, axial_preview_df, axial_stats = build_axial_plan(e9_df)
metadata_e9_row = plan_row("metadata_e9", E9_SPLIT_CSV, E9_SPLIT_CSV.parent, METADATA_DEST, sha256=sha256_stream(E9_SPLIT_CSV))
metadata_e9_row["relative_path"] = "E9_t2_final_labels_curated_split.csv"
metadata_e9_row["destination_uri"] = gcs_join(METADATA_DEST, "E9_t2_final_labels_curated_split.csv")

print("AXIAL stats:", axial_stats)
display(axial_preview_df.head(10))
display(axial_plan_df.head(10))


Resolver axial: fallback notebook
E9 filas: 610
E9 duplicados exactos de pares: 0


,split,rows,patients
0,test,102,29
1,train,427,128
2,val,81,27


AXIAL stats: {'paths_total': 1220, 'paths_rebased': 0, 'paths_not_rebased': 1220, 'problems': {}}


,row,component,raw_path,resolved_path,was_rebased
0,0,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
1,0,axial_mask,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
2,1,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
3,1,axial_mask,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
4,2,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
5,2,axial_mask,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
6,3,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
7,3,axial_mask,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
8,4,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False
9,4,axial_mask,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False


,component,source_path,source_root,relative_path,destination_uri,suffix,size_bytes,modified_at,sha256,referenced_rows,exists,is_symlink,status
0,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207458,2026-06-27T18:44:18Z,None,1,True,False,OK
1,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207460,2026-06-27T18:44:18Z,None,1,True,False,OK
2,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207456,2026-06-27T18:44:18Z,None,1,True,False,OK
3,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207232,2026-06-27T18:44:19Z,None,1,True,False,OK
4,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207232,2026-06-27T18:44:19Z,None,1,True,False,OK
5,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207224,2026-06-27T18:44:19Z,None,1,True,False,OK
6,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207644,2026-06-27T18:44:21Z,None,1,True,False,OK
7,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207644,2026-06-27T18:44:21Z,None,1,True,False,OK
8,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207638,2026-06-27T18:44:21Z,None,1,True,False,OK
9,axial_image,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,extracted/_nested/main_dataset__MRI_Data/01_MR...,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.ima,207282,2026-06-27T18:44:22Z,None,1,True,False,OK


## Bootstrap sagital opcional


In [9]:
def inspect_bootstrap_model(path: Path) -> dict[str, Any]:
    row = {
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
        "modified_at": file_modified_at(path),
        "sha256": None,
        "included": INCLUDE_BOOTSTRAP_MODEL,
        "status": "PRESENT" if path.is_file() else "ABSENT",
    }
    if path.is_file() and (INCLUDE_BOOTSTRAP_MODEL or COMPUTE_FULL_SHA256):
        row["sha256"] = sha256_stream(path)
    return row

bootstrap_info = inspect_bootstrap_model(SAGITTAL_BOOTSTRAP_MODEL)
print("Bootstrap sagital:", bootstrap_info)
if INCLUDE_BOOTSTRAP_MODEL and SAGITTAL_BOOTSTRAP_MODEL.is_file():
    bootstrap_row = plan_row("bootstrap_model", SAGITTAL_BOOTSTRAP_MODEL, SAGITTAL_BOOTSTRAP_MODEL.parent, BOOTSTRAP_DEST, sha256=bootstrap_info["sha256"])
    bootstrap_row["relative_path"] = "sagittal_spider_multiclass_final_best.pt"
    bootstrap_row["destination_uri"] = gcs_join(BOOTSTRAP_DEST, "sagittal_spider_multiclass_final_best.pt")
else:
    bootstrap_row = None
    print("SKIP: el JSON E5 es la fuente primaria y el bootstrap es opcional.")


Bootstrap sagital: {'exists': True, 'size_bytes': 1975947, 'modified_at': '2026-06-27T14:30:47Z', 'sha256': None, 'included': False, 'status': 'PRESENT'}
SKIP: el JSON E5 es la fuente primaria y el bootstrap es opcional.


## Manifest combinado y validaciones globales


In [10]:
def maybe_fill_dataset_sha(df: pd.DataFrame) -> pd.DataFrame:
    if not COMPUTE_FULL_SHA256 or df.empty:
        return df
    out = df.copy()
    out["sha256"] = [sha256_stream(Path(p)) if Path(p).is_file() else None for p in out["source_path"]]
    return out

manifest_parts = [
    maybe_fill_dataset_sha(spider_images_df),
    maybe_fill_dataset_sha(spider_masks_df),
    maybe_fill_dataset_sha(axial_plan_df),
    pd.DataFrame([metadata_e5_row, metadata_e9_row]),
]
if bootstrap_row is not None:
    manifest_parts.append(pd.DataFrame([bootstrap_row]))

manifest_df = pd.concat(manifest_parts, ignore_index=True).fillna(value=pd.NA)

def validate_manifest(manifest: pd.DataFrame) -> dict[str, Any]:
    errors = []
    warnings = []
    if manifest.empty:
        errors.append("manifest vacio")
    duplicated_dest = manifest[manifest["destination_uri"].duplicated(keep=False)]["destination_uri"].dropna().unique().tolist()
    if duplicated_dest:
        errors.append(f"destination_uri duplicados: {duplicated_dest[:10]}")
    duplicated_source_component = manifest[manifest.duplicated(["component", "source_path"], keep=False)][["component", "source_path"]].head(10).to_dict("records")
    if duplicated_source_component:
        errors.append(f"source_path duplicado por componente: {duplicated_source_component}")
    for column, label in [("exists", "faltantes"), ("is_symlink", "symlinks")]:
        count = int((manifest[column] == (column == "is_symlink")).sum()) if column == "is_symlink" else int((manifest[column] == False).sum())
        if count:
            errors.append(f"{label}: {count}")
    zero_size = int((manifest["size_bytes"].fillna(0) == 0).sum())
    if zero_size:
        errors.append(f"tamano cero: {zero_size}")
    invalid_dest = [uri for uri in manifest["destination_uri"].dropna().astype(str) if not uri.startswith(GCS_BUCKET_URI.rstrip("/") + "/")]
    if invalid_dest:
        errors.append(f"destinos fuera del bucket: {invalid_dest[:10]}")
    zip_rows = manifest[manifest["suffix"].astype(str).str.lower().eq(".zip")]
    if len(zip_rows):
        errors.append("hay ZIP incluidos")
    hidden = manifest[manifest["relative_path"].astype(str).str.contains(r"(^|/)\.", regex=True, na=False)]
    if len(hidden):
        warnings.append(f"archivos ocultos inesperados: {len(hidden)}")
    parent_refs = manifest[manifest["relative_path"].astype(str).str.contains("..", regex=False, na=False)]
    if len(parent_refs):
        errors.append("relative_path contiene '..'")
    return {"errors": errors, "warnings": warnings, "duplicated_destinations": duplicated_dest}

manifest_validation = validate_manifest(manifest_df)
print("Manifest filas:", len(manifest_df))
print("Manifest validation:", manifest_validation)
display(manifest_df.head(20))


Manifest filas: 2058
Manifest validation: {'errors': [], 'warnings': [], 'duplicated_destinations': []}


/tmp/ipykernel_2017/1457976780.py:43: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hidden = manifest[manifest["relative_path"].astype(str).str.contains(r"(^|/)\.", regex=True, na=False)]


,component,source_path,source_root,relative_path,destination_uri,suffix,size_bytes,modified_at,sha256,referenced_rows,exists,is_symlink,status
0,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,100_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,7105555,2022-11-01T15:19:38Z,<NA>,<NA>,True,False,OK
1,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,100_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,7118571,2022-11-01T15:19:36Z,<NA>,<NA>,True,False,OK
2,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,101_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,1839721,2022-11-01T15:12:26Z,<NA>,<NA>,True,False,OK
3,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,101_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2361432,2022-11-01T15:12:24Z,<NA>,<NA>,True,False,OK
4,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,104_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,1822712,2022-11-01T15:08:20Z,<NA>,<NA>,True,False,OK
5,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,104_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2706068,2022-11-01T15:08:18Z,<NA>,<NA>,True,False,OK
6,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,105_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,4534255,2022-11-01T15:03:52Z,<NA>,<NA>,True,False,OK
7,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,105_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,4604875,2022-11-01T15:03:52Z,<NA>,<NA>,True,False,OK
8,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,106_t1.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2971403,2022-11-01T15:13:34Z,<NA>,<NA>,True,False,OK
9,spider_image,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,106_t2.mha,gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/...,.mha,2912377,2022-11-01T15:13:34Z,<NA>,<NA>,True,False,OK


## Resumen de transferencia


In [11]:
def summarize_manifest(manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for component, group in manifest.groupby("component", dropna=False):
        missing_count = int((group["exists"] == False).sum())
        zero_size_count = int((group["size_bytes"].fillna(0) == 0).sum())
        symlink_count = int((group["is_symlink"] == True).sum())
        total_size = int(group["size_bytes"].fillna(0).sum())
        rows.append({
            "component": component,
            "file_count": len(group),
            "total_size_bytes": total_size,
            "total_size_gib": total_size / (1024 ** 3),
            "missing_count": missing_count,
            "zero_size_count": zero_size_count,
            "symlink_count": symlink_count,
            "readiness": "READY" if missing_count == 0 and zero_size_count == 0 and symlink_count == 0 else "NOT_READY",
        })
    return pd.DataFrame(rows).sort_values("component")

summary_df = summarize_manifest(manifest_df)
spider_size = int(summary_df[summary_df["component"].isin(["spider_image", "spider_mask"])] ["total_size_bytes"].sum())
axial_size = int(summary_df[summary_df["component"].isin(["axial_image", "axial_mask"])] ["total_size_bytes"].sum())
print("Total archivos:", int(summary_df["file_count"].sum()))
print("Total bytes:", int(summary_df["total_size_bytes"].sum()))
print("Total GiB:", float(summary_df["total_size_gib"].sum()))
print("SPIDER bytes:", spider_size)
print("AXIAL subset bytes:", axial_size)
print("Filas E9:", len(e9_df))
print("Archivos unicos axiales:", len(axial_plan_df))
print("Advertencia: Este tamano corresponde al subset requerido por el CSV E9, no al contenido total de AXIAL_ALKAFRI.")
display(summary_df)


Total archivos: 2058
Total bytes: 3922288649
Total GiB: 3.652915962971747
SPIDER bytes: 3794045915
AXIAL subset bytes: 127819918
Filas E9: 610
Archivos unicos axiales: 1162
Advertencia: Este tamano corresponde al subset requerido por el CSV E9, no al contenido total de AXIAL_ALKAFRI.


,component,file_count,total_size_bytes,total_size_gib,missing_count,zero_size_count,symlink_count,readiness
0,axial_image,610,127018778,1.182955e-01,0,0,0,READY
1,axial_mask,552,801140,7.461198e-04,0,0,0,READY
2,metadata_e5,1,222,2.067536e-07,0,0,0,READY
3,metadata_e9,1,422594,3.935713e-04,0,0,0,READY
4,spider_image,447,3707013638,3.452425e+00,0,0,0,READY
5,spider_mask,447,87032277,8.105512e-02,0,0,0,READY


## Plan futuro de transferencia, sin ejecucion

Estrategia futura recomendada:

- usar `gcs_upload_manifest.csv` como fuente de verdad;
- transferir SPIDER desde sus dos carpetas extraidas;
- transferir AXIAL solo desde el subset manifestado por E9, preservando la jerarquia posterior a `AXIAL_ALKAFRI/`;
- transferir metadata como archivos individuales;
- incluir el bootstrap sagital solo si se decide activar `INCLUDE_BOOTSTRAP_MODEL`;
- no copiar ZIP ni archivos auxiliares;
- no copiar la raiz completa de AXIAL, porque incluiria archivos no requeridos por el split curado.

Este notebook no ejecuta ningun comando de nube ni prepara credenciales.


## Quality gate final


In [12]:
SPIDER_READY = bool(spider_pairing["ready"]) and not spider_images_df.empty and not spider_masks_df.empty and not any(spider_images_df["status"] != "OK") and not any(spider_masks_df["status"] != "OK")
AXIAL_READY = not axial_plan_df.empty and not axial_stats["problems"] and not any(axial_plan_df["status"] != "OK")
METADATA_READY = metadata_e5_row["status"] == "OK" and metadata_e9_row["status"] == "OK" and not manifest_validation["duplicated_destinations"]
BOOTSTRAP_INCLUDED = bool(INCLUDE_BOOTSTRAP_MODEL and bootstrap_row is not None)
GLOBAL_READY_FOR_UPLOAD = bool(SPIDER_READY and AXIAL_READY and METADATA_READY and not manifest_validation["errors"])

gate = {
    "SPIDER_READY": SPIDER_READY,
    "AXIAL_READY": AXIAL_READY,
    "METADATA_READY": METADATA_READY,
    "BOOTSTRAP_INCLUDED": BOOTSTRAP_INCLUDED,
    "GLOBAL_READY_FOR_UPLOAD": GLOBAL_READY_FOR_UPLOAD,
    "errors": manifest_validation["errors"],
    "warnings": manifest_validation["warnings"],
}
print(json.dumps(gate, ensure_ascii=False, indent=2))
if not GLOBAL_READY_FOR_UPLOAD:
    print("No listo para upload: revisar errores accionables antes de preparar transferencia real.")


{
  "SPIDER_READY": true,
  "AXIAL_READY": true,
  "METADATA_READY": false,
  "BOOTSTRAP_INCLUDED": false,
  "GLOBAL_READY_FOR_UPLOAD": false,
  "errors": [],
  "warnings": []
}
No listo para upload: revisar errores accionables antes de preparar transferencia real.


## Escritura opcional del plan


In [13]:
def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.parent / f".{path.name}.{os.getpid()}.tmp"
    with open(tmp, "w", encoding="utf-8", newline="") as fh:
        fh.write(text)
        fh.flush()
        os.fsync(fh.fileno())
    os.replace(tmp, path)


def atomic_write_dataframe_csv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.parent / f".{path.name}.{os.getpid()}.tmp"
    with open(tmp, "w", encoding="utf-8", newline="") as fh:
        df.to_csv(fh, index=False)
        fh.flush()
        os.fsync(fh.fileno())
    os.replace(tmp, path)


def plan_readme_text() -> str:
    return f"""# GCS dataset migration plan\n\nGenerated UTC: {utc_now()}\n\nNo upload was performed.\n\nSources:\n- SPIDER images: {SPIDER_IMAGES_SOURCE}\n- SPIDER masks: {SPIDER_MASKS_SOURCE}\n- AXIAL root: {AXIAL_ROOT}\n- E5 mapping: {E5_MAPPING_JSON}\n- E9 split: {E9_SPLIT_CSV}\n\nDestination bucket: {GCS_BUCKET_URI}\n\nFlags:\n- WRITE_PLAN_FILES={WRITE_PLAN_FILES}\n- COMPUTE_FULL_SHA256={COMPUTE_FULL_SHA256}\n- INCLUDE_BOOTSTRAP_MODEL={INCLUDE_BOOTSTRAP_MODEL}\n\nReadiness:\n- SPIDER_READY={SPIDER_READY}\n- AXIAL_READY={AXIAL_READY}\n- METADATA_READY={METADATA_READY}\n- GLOBAL_READY_FOR_UPLOAD={GLOBAL_READY_FOR_UPLOAD}\n\nProblems:\n{json.dumps(manifest_validation, ensure_ascii=False, indent=2)}\n"""

if WRITE_PLAN_FILES:
    PLAN_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_dataframe_csv(PLAN_ROOT / "gcs_upload_manifest.csv", manifest_df)
    atomic_write_text(PLAN_ROOT / "gcs_upload_manifest.json", manifest_df.to_json(orient="records", indent=2, force_ascii=False))
    atomic_write_dataframe_csv(PLAN_ROOT / "gcs_upload_summary.csv", summary_df)
    atomic_write_text(PLAN_ROOT / "gcs_upload_validation.json", json.dumps(manifest_validation, ensure_ascii=False, indent=2))
    atomic_write_dataframe_csv(PLAN_ROOT / "axial_e9_resolved_paths_preview.csv", axial_preview_df.head(200))
    atomic_write_text(PLAN_ROOT / "README.md", plan_readme_text())
    print(f"Plan escrito en {PLAN_ROOT}")
else:
    print("WRITE_PLAN_FILES=False: no se crea PLAN_ROOT ni se escribe ningun archivo.")


WRITE_PLAN_FILES=False: no se crea PLAN_ROOT ni se escribe ningun archivo.
